In [20]:
import collections
import re
from d2l import torch as d2l

In [21]:
## read text data
import re
from d2l import torch as d2l

#@save
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt','090b5e7e70c295757f55df93cb0a180b9691891a')
#@save
def read_time_machine(): 
   with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
        
   
    # 1. re.sub('[^A-Za-z]+', ' ', line)：将非字母字符替换为空格
    # 2. .strip()：去除首尾空白符
    # 3. .lower()：统一转换为小写
        return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

if __name__ == '__main__':
    lines = read_time_machine()
    
    print(f'# 文本总行数: {len(lines)}')
    print('第一行')
    print(lines[0])
    print('第十一行')
    print(lines[10])

# 文本总行数: 3221
第一行
the time machine by h g wells
第十一行
twinkled and his usually pale face was flushed and animated the


In [22]:
## tokenizer
def tokenize(lines,token='word'):
    if token == 'word':
        return [line.split() for line in lines]

    elif token == 'char':
        return [list(line) for line in lines]
    else :
        print('error: unknown token type'+ token)

tokens = tokenize(lines)

for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


In [23]:
## 统计token出现频率
def count_corpus(tokens):
    ## 判断tokens的shape
    if len(tokens) == 0 or isinstance(tokens[0],list):
        flat_list = []
        for line in tokens:
            for token in line:
                flat_list.append(token)
        tokens = flat_list
    return collections.Counter(tokens)

In [24]:
class Vocab:
    def __init__(self,tokens=None,min_freq=0,reserved_tokens=None):
        if tokens == None:
            token = []
        if reserved_tokens == None:
            reserved_tokens = []
        ## 计算出现频率
        counter = count_corpus(tokens)
        self._token_freqs = sorted(
            counter.items(),key = lambda x:x[1],reverse=True
        )
        ## index to token用list
        self.idx_to_token = ['<unk>'] + reserved_tokens

        ## token to index 用 dict
        self.token_to_idx = {
            token:idx for idx,token in enumerate(self.idx_to_token)
        }

        for token,freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) -1
    def __len__(self):
            return len(self.idx_to_token)
    def __getitem__(self,tokens):
        if not isinstance(tokens,(list,tuple)):
            return self.token_to_idx.get(tokens,self.unk)
        return [self.__getitem__(token) for token in tokens]
    def to_tokens(self,indices):
        if not isinstance(indices,(list,tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]

    @property
    def unk(self):
        return 0

    @property 
    def token_freqs(self):
        return self._token_freqs

In [26]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


In [28]:
def load_corpus_time_machine(max_tokens = -1):
    lines = read_time_machine()
    tokens = tokenize(lines,'char')
    vocab = Vocab(tokens)
    corpus = []
    for line in tokens:
        for token in line:
            index = vocab[token]
            corpus.append(index)
    if max_tokens > 0 :
        corpus = corpus[:max_tokens]
    return corpus,vocab


In [29]:
corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

(170580, 28)